# 사출성형기 baseline: 직접 실행하기

커널은 **Python (kamp-base)**를 선택하고 위에서 아래로 `Shift+Enter`로 실행하세요.

현재는 **저장된 결과 확인 모드**입니다. 새로 학습하려면 아래 설정 셀의 `RUN_TRAINING = True`로 바꾸세요. 원본 데이터와 기존 baseline 결과는 덮어쓰지 않습니다.

양성 클래스는 숫자 `1`이며 **불량이라는 의미는 아직 확인되지 않았습니다.** AP·F1은 class_1에 대한 지표입니다.

학습 구현은 `scripts/train_baseline.py`, 평가 그림 구현은 `scripts/report_baseline.py`에 있습니다. 노트북과 CLI가 같은 코드를 사용합니다.

## 1. 커널 및 프로젝트 경로 확인

In [ ]:
import sys
import json
import os
import subprocess
from pathlib import Path
from datetime import datetime
from uuid import uuid4

import pandas as pd
from IPython.display import display, Image, Markdown

# 프로젝트 루트, 과제 폴더, notebooks 폴더 어디서 시작해도 찾습니다.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "1.injection-molding" / "configs" / "baseline_v1.json").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("kamp-ai 폴더에서 Jupyter를 실행하세요.")
PROJECT = ROOT / "1.injection-molding"
print("Python:", sys.version.split()[0])
print("커널 실행 경로:", sys.executable)
print("프로젝트:", PROJECT)
if Path(sys.prefix).name != "kamp-base":
    raise RuntimeError("노트북 커널을 Python (kamp-base)로 변경한 후 다시 실행하세요.")

## 2. 데이터와 라벨 분포 확인
ID와 `PassOrFail`은 공정 입력이 아닙니다. `Clamp_Open_Position`은 라벨 데이터에서 상수여서 baseline 입력에서 제외합니다.

In [ ]:
datasets = {
    group: pd.read_csv(PROJECT / "dataset" / f"moldset_labeled_{group}.csv")
    for group in ["cn7", "rg3"]
}
display(pd.DataFrame([
    {"product_group": group, "rows": len(df),
     "class_0": int((df.PassOrFail == 0).sum()),
     "class_1": int((df.PassOrFail == 1).sum())}
    for group, df in datasets.items()
]))
display(datasets["cn7"].head())

## 3. 중복·충돌 직접 확인
공정값이 같은 행을 하나의 그룹으로 묶습니다. 라벨이 다른 행을 삭제하거나 수정하지 않습니다.

In [ ]:
duplicate_summary = []
for group, df in datasets.items():
    features = [c for c in df.columns if c not in ["Unnamed: 0", "PassOrFail"]]
    grouped = df.groupby(features, dropna=False)["PassOrFail"].agg(["size", "nunique"])
    duplicate_summary.append({
        "product_group": group,
        "unique_patterns": len(grouped),
        "duplicate_rows_beyond_first": int(df[features].duplicated().sum()),
        "conflicting_groups": int((grouped["nunique"] > 1).sum()),
    })
display(pd.DataFrame(duplicate_summary))

## 4. 실행 모드와 설정

- `RUN_TRAINING = False`: 이미 완료된 `baseline_v1` 결과만 읽습니다.
- `RUN_TRAINING = True`: 아래 학습 셀에서 새 실험을 실행합니다. 매 실행마다 고유 폴더가 생성됩니다.
- 설정을 바꾸려면 이 셀에서 수정합니다. 원본 설정 JSON은 변경하지 않습니다.
- 예: `config["xgboost"]["max_depth"] = 2`
- 모든 scenario와 기본 모델 목록은 유지하면 보고서의 기본 비교 그림도 생성됩니다. 다른 scenario/모델 구성을 시험할 경우 보고서 코드도 그 구성에 맞춰야 합니다.

학습은 CPU로 실행합니다. 동일 그룹은 외부 평가뿐 아니라 내부 임계값 선택에서도 분리합니다.

In [ ]:
RUN_TRAINING = False
EXISTING_RUN = PROJECT / "artifacts" / "baseline_v1"
config = json.loads((PROJECT / "configs" / "baseline_v1.json").read_text())

# 필요하면 아래 주석을 해제해 새 실험 설정을 바꾸세요.
# config["xgboost"]["max_depth"] = 2
# config["xgboost"]["n_estimators"] = 200

display(config)

## 5. 학습 실행 또는 기존 결과 선택

새 학습 시 외부 3-fold × 3 seeds × 3 scenarios × 5 모델 설정을 비교합니다.
XGBoost와 Logistic은 각 외부 학습 fold 내부의 그룹 OOF로 F1 임계값을 선택합니다.

`sys.executable`로 현재 노트북과 같은 Python을 사용합니다. 진행 상황은 셀에, 전체 출력은 실험 폴더의 `training.log`에 저장합니다.
학습 셀을 중단하면 실행 중인 자식 프로세스도 종료합니다. 이후 결과는 `manifest.json`의 complete 상태를 검사합니다.

In [ ]:
def run_script(script, arguments, log_path):
    command = [sys.executable, "-u", str(PROJECT / "scripts" / script), *map(str, arguments)]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    with log_path.open("w") as log:
        process = subprocess.Popen(
            command, cwd=ROOT, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True,
        )
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                if line.startswith("Completed") or "Warning" in line:
                    print(line.rstrip(), flush=True)
            return_code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
    if return_code:
        print(log_path.read_text()[-6000:])
        raise RuntimeError(f"실행 실패: {script}; 로그: {log_path}")

if RUN_TRAINING:
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
    experiment_dir = PROJECT / "artifacts" / "notebook_runs" / run_id
    experiment_dir.mkdir(parents=True, exist_ok=False)
    RUN_DIR = experiment_dir / "results"
    config_path = experiment_dir / "config.json"
    config_path.write_text(json.dumps(config, indent=2) + "\n")
    run_script("train_baseline.py", ["--config", config_path, "--output-dir", RUN_DIR],
               experiment_dir / "training.log")
    run_script("report_baseline.py", ["--run-dir", RUN_DIR], experiment_dir / "report.log")
else:
    RUN_DIR = EXISTING_RUN

manifest = json.loads((RUN_DIR / "manifest.json").read_text())
if manifest["status"] != "complete":
    raise RuntimeError("미완료 실행입니다. training.log를 확인하세요.")
print("선택한 결과:", RUN_DIR)
print("실제 실행 설정:")
display(json.loads((RUN_DIR / "config.json").read_text()))
display(manifest)

## 6. 모델 비교표

`inner_f1`은 평가 정답을 보지 않고 학습 데이터 내부에서 선택한 임계값입니다.
AP와 Recall@10%는 임계값과 무관합니다. 평균±표준편차는 9개 외부 fold에 대한 값이며 신뢰구간이 아닙니다.

다른 seed나 모델을 반복 선택하며 이 CV에 맞추면 과적합할 수 있습니다.

In [ ]:
summary = pd.read_csv(RUN_DIR / "summary.csv")
POLICY = "inner_f1"  # "fixed_0.5"와 비교할 수 있습니다.
columns = ["scenario", "model", "ap_mean", "ap_std", "f1_mean", "f1_std",
           "precision_mean", "recall_mean", "recall_at_10pct_mean", "brier_mean"]
display(summary.loc[summary.policy == POLICY, columns].round(4))
display(Image(filename=str(RUN_DIR / "comparison.png")))

## 7. fold별 변동과 임계값 확인
평균만 보지 말고 fold별 class_1 수와 임계값 변동을 함께 확인합니다.

In [ ]:
fold_metrics = pd.read_csv(RUN_DIR / "fold_metrics.csv")
SCENARIO = "cn7"  # "rg3", "pooled"
MODEL = "xgboost_balanced"  # "dummy_prior", "logistic", "logistic_balanced", "xgboost"
selected = fold_metrics[
    (fold_metrics.scenario == SCENARIO) & (fold_metrics.model == MODEL)
    & (fold_metrics.policy == POLICY)
]
display(selected[["seed", "fold", "positives", "selected_threshold", "ap", "f1",
                  "precision", "recall", "tp", "fp", "fn", "tn", "recall_at_10pct"]].round(4))

## 8. 실제 평가 예측과 오류 행 확인

한 반복의 OOF만 선택해 같은 행을 중복 집계하지 않습니다. 각 행은 자신이 학습에 포함되지 않은 외부 모델로 예측됐습니다.
라벨 충돌 여부도 함께 보되 모든 오류를 라벨 오류라고 단정하지 않습니다.

In [ ]:
oof = pd.read_csv(RUN_DIR / "oof_predictions.csv")
SEED = 42
part = oof[(oof.scenario == SCENARIO) & (oof.model == MODEL) & (oof.seed == SEED)].copy()
prediction_column = "prediction_inner_f1" if POLICY == "inner_f1" else "prediction_fixed"
print("선택한 평가 행 수:", len(part))
display(pd.crosstab(part.PassOrFail, part[prediction_column],
                    rownames=["실제 class"], colnames=["예측 class"]).reindex(index=[0, 1], columns=[0, 1], fill_value=0))
errors = part[part.PassOrFail != part[prediction_column]]
display(errors.sort_values("p_class_1", ascending=False).head(20))

## 9. 결과 보고서
전체 결과와 제한사항을 확인합니다. 미라벨 추론·최종 배포 모델·확률 보정은 별도 단계입니다.

In [ ]:
display(Markdown((RUN_DIR / "report.md").read_text()))